# Topic 06 — Practice: Merge / Join / Concat

**The notebook is the lesson.** Work top to bottom. Cells with `assert` grade themselves — a silent run = pass, an `AssertionError` = fix your code. Warm-Up, Interview Lens and Reflection have **no answer key** — answer in writing.

_Spaced repetition: ~60% of this notebook is the current topic, ~40% revisits earlier topics._

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
oi = items.merge(orders[['order_id','channel','customer_id']], on='order_id', how='left')
customers = pd.read_csv(RAW+'customers.csv', dtype={'customer_id':str}).drop_duplicates('customer_id')
products['list_price'] = np.where(products['list_price']<0, np.nan, products['list_price'])


## 🔁 Warm-Up Recall (earlier topics — no answers given)

From **Topics 01–05**:

1. (T05) What does `transform` give you that `agg` doesn't?
2. (T04) Why coerce negative prices to NaN before joining cost data?
3. NumPy: how do you test membership of one array's values in another?

In [ ]:
# NumPy recall: a 'join audit' on key arrays
left = np.array([1,2,3,4,5]); right = np.array([3,4,5,6])
matched = ...   # in both
orphans_n = ...   # in left, not in right
assert sorted(matched) == [3,4,5] and orphans_n == 2
print('ok')

## 🎯 Core Lesson Tasks (current topic)

A `left` join must not change the left row count — unless the right key has duplicates. Validate every join.

**Core 1 — safe left join.** Bring `unit_cost` into `items` (validate many_to_one). Row count must not change.

In [ ]:
before = len(items)
m = ...
assert len(m) == before
print('rows preserved:', len(m))

**Core 2 — line profit.** On `m`, compute `line_profit = line_revenue - quantity*unit_cost`. Count lines missing cost.

In [ ]:
m['line_profit'] = ...
missing_cost = ...
assert 'line_profit' in m.columns
print('missing cost:', missing_cost)

**Core 3 — find orphans.** Left-merge `orders`→`customers` with `indicator=True`; isolate `left_only`.

In [ ]:
om = orders.merge(customers[['customer_id']], on='customer_id', how='left', indicator=True)
orphans = ...
assert orphans.shape[0] > 0
print('orphan orders:', orphans.shape[0])

## 🔀 Mixed Review Tasks (earlier topics — spaced repetition)

**Mixed review (Topic 05) — group the joined table.** Profit per channel from the costed master table.

In [ ]:
master = items.merge(products[['product_id','unit_cost']], on='product_id', how='left').merge(orders[['order_id','channel']], on='order_id', how='left')
master['line_profit'] = master['line_revenue'] - master['quantity']*master['unit_cost']
profit_by_channel = ...
assert len(master)==len(items)
profit_by_channel.sort_values(ascending=False)

**Mixed review (Topic 03) — mask the result.** From the costed master, how many lines are loss-making (`line_profit < 0`)?

In [ ]:
n_loss = ...
assert n_loss >= 0
print(n_loss)

## 🕵️ Data Detective Investigation

**Case file #6 — true profit, joined safely.** Revenue lied because cost lived in another table. Build the master table (items→products→orders), compute profit per channel, and prove no join changed the grain.

In [ ]:
master = (items.merge(products[['product_id','unit_cost']], on='product_id', how='left', validate='many_to_one')
              .merge(orders[['order_id','channel']], on='order_id', how='left', validate='many_to_one'))
master['line_profit'] = master['line_revenue'] - master['quantity']*master['unit_cost']
assert len(master) == len(items), 'a join changed the grain!'
print(master.groupby('channel')['line_profit'].sum().sort_values(ascending=False))

## 🔎 Interview Lens (write answers — no model answers)

- **Q18:** Walk through inner/left/right/outer with two Aurora tables — who's lost or invented?
- **Q19:** What risks exist when merging? How do you detect a many-to-many blow-up *before* it corrupts totals?
- **Q20:** After a left merge your row count went *up*. What happened and how do you prove it?

## ✍️ Reflection (write short explanations)

1. Answer Q18 and Q19 in writing.
2. Did any join change your row count? What would that do to revenue?
3. **Investigation log:** what is *profit* (not revenue) telling you now?